# FinTech Transaction Data Analysis

This beginner-friendly notebook analyzes the real `transactions_data.csv` file. It contains millions of rows, so we read it in chunks.

We will clean missing data, study amounts and frequency, find high-value transactions, compare customers and categories, and plot trends.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
DATA_PATH = Path('data/transactions_data.csv')
if not DATA_PATH.exists(): DATA_PATH = Path('data/transactions_data.csv')
print('Using:', DATA_PATH.resolve())
print(f'File size: {DATA_PATH.stat().st_size / 1_000_000_000:.2f} GB')

## 1. Inspect the source file

Always check the first few rows before analyzing a new dataset.

In [ ]:
preview = pd.read_csv(DATA_PATH, nrows=5)
display(preview)
print('Columns:', preview.columns.tolist())
print('Example amount values:', preview['amount'].tolist())

## 2. Clean the data

Amounts contain `$` signs. MCC is Merchant Category Code. We map common codes to simple categories and keep the rest as `Other`.

In [ ]:
USE_COLS = ['id','date','client_id','card_id','amount','use_chip','merchant_id','merchant_city','merchant_state','zip','mcc','errors']
CHUNK_SIZE = 250_000
category_map = {5411:'Groceries',5812:'Dining',5813:'Dining',5814:'Dining',5311:'Shopping',5621:'Shopping',5651:'Shopping',5541:'Transport',5542:'Transport',4900:'Bills',4814:'Bills',4511:'Travel',4722:'Travel',7011:'Travel',5912:'Healthcare',5975:'Healthcare'}

def clean_chunk(chunk):
    chunk['amount'] = chunk['amount'].astype('string').str.replace('$','',regex=False).str.replace(',','',regex=False)
    chunk['amount'] = pd.to_numeric(chunk['amount'], errors='coerce')
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
    chunk['mcc'] = pd.to_numeric(chunk['mcc'], errors='coerce')
    chunk['category'] = chunk['mcc'].map(category_map).fillna('Other')
    chunk['month'] = chunk['date'].dt.to_period('M').astype('string')
    chunk['weekday'] = chunk['date'].dt.day_name()
    chunk['hour'] = chunk['date'].dt.hour
    return chunk

display(clean_chunk(preview.copy()))

## 3. Missing-data report

This counts missing values in the original file without loading all rows at once.

In [ ]:
missing = pd.Series(0, index=USE_COLS, dtype='int64')
raw_rows = 0
for chunk in pd.read_csv(DATA_PATH, usecols=USE_COLS, chunksize=CHUNK_SIZE):
    missing = missing.add(chunk.isna().sum(), fill_value=0)
    raw_rows += len(chunk)
missing_report = pd.DataFrame({'missing_count':missing.astype(int), 'missing_percent':(missing/raw_rows*100).round(2)}).sort_values('missing_count', ascending=False)
print(f'Total source rows: {raw_rows:,}')
display(missing_report)

## 4. Analyze all rows in chunks

We save summary tables and a small sample for charts. Rows missing a date, client ID, or valid non-negative amount are excluded from spending calculations.

In [ ]:
monthly_parts=[]; category_parts=[]; customer_parts=[]; weekday_parts=[]; hour_parts=[]; method_parts=[]; samples=[]; amount_samples=[]; valid_rows=0
for chunk in pd.read_csv(DATA_PATH, usecols=USE_COLS, chunksize=CHUNK_SIZE):
    chunk = clean_chunk(chunk)
    clean = chunk.dropna(subset=['date','client_id','amount']).copy()
    clean = clean[clean['amount'] >= 0]
    valid_rows += len(clean)
    monthly_parts.append(clean.groupby('month').agg(total_spend=('amount','sum'),transactions=('id','count')).reset_index())
    category_parts.append(clean.groupby('category').agg(total_spend=('amount','sum'),transactions=('id','count'),amount_sum=('amount','sum')).reset_index())
    customer_parts.append(clean.groupby('client_id').agg(total_spend=('amount','sum'),transactions=('id','count'),largest_transaction=('amount','max')).reset_index())
    weekday_parts.append(clean.groupby('weekday').agg(total_spend=('amount','sum'),transactions=('id','count')).reset_index())
    hour_parts.append(clean.groupby('hour').size().rename('transactions').reset_index())
    method_parts.append(clean.groupby('use_chip').agg(total_spend=('amount','sum'),transactions=('id','count')).reset_index())
    samples.append(clean.sample(n=min(5000,len(clean)), random_state=42))
    amount_samples.append(clean['amount'].sample(n=min(5000,len(clean)), random_state=42))

def combine(parts, group_col):
    x = pd.concat(parts, ignore_index=True)
    if group_col == 'client_id': return x.groupby(group_col).agg(total_spend=('total_spend','sum'),transactions=('transactions','sum'),largest_transaction=('largest_transaction','max'))
    return x.groupby(group_col).agg(total_spend=('total_spend','sum'),transactions=('transactions','sum'))

monthly=combine(monthly_parts,'month').sort_index()
category_summary=combine(category_parts,'category').sort_values('total_spend',ascending=False)
customer_summary=combine(customer_parts,'client_id').sort_values('total_spend',ascending=False)
weekday_summary=combine(weekday_parts,'weekday')
hour_summary=pd.concat(hour_parts).groupby('hour')['transactions'].sum().sort_index()
method_summary=combine(method_parts,'use_chip').sort_values('total_spend',ascending=False)
chart_data=pd.concat(samples,ignore_index=True)
amount_sample=pd.concat(amount_samples,ignore_index=True)
print(f'Valid rows used: {valid_rows:,}')
display(monthly.head())

## 5. Amounts, frequency, customers, and categories

In [ ]:
display(amount_sample.describe().to_frame('amount').style.format('${:,.2f}'))
print('Top customers by spend')
display(customer_summary.head(10).style.format({'total_spend':'${:,.2f}','largest_transaction':'${:,.2f}'}))
print('Categories by spend')
display(category_summary.style.format({'total_spend':'${:,.2f}'}))

## 6. High-value transactions

The 95th percentile is a simple first-pass threshold. It flags unusually large amounts, but it does not prove fraud.

In [ ]:
threshold = amount_sample.quantile(.95)
high_value = chart_data[chart_data['amount'] > threshold].sort_values('amount',ascending=False)
print(f'Approximate 95th-percentile threshold: ${threshold:,.2f}')
display(high_value[['id','date','client_id','amount','category','merchant_city','use_chip']].head(15).style.format({'amount':'${:,.2f}'}))

## 7. Spending patterns and visualizations

In [ ]:
order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_summary=weekday_summary.reindex(order)
fig,axes=plt.subplots(2,2,figsize=(16,10))
sns.barplot(data=category_summary.reset_index(),y='category',x='total_spend',ax=axes[0,0],color='steelblue'); axes[0,0].set_title('Total spend by category'); axes[0,0].set_xlabel('Spend ($)')
sns.barplot(data=weekday_summary.reset_index(),x='weekday',y='total_spend',ax=axes[0,1],color='seagreen'); axes[0,1].set_title('Total spend by weekday'); axes[0,1].tick_params(axis='x',rotation=35)
sns.lineplot(x=hour_summary.index,y=hour_summary.values,marker='o',ax=axes[1,0],color='darkorange'); axes[1,0].set_title('Transaction frequency by hour'); axes[1,0].set_xlabel('Hour of day'); axes[1,0].set_ylabel('Transactions')
sns.barplot(data=method_summary.reset_index(),y='use_chip',x='transactions',ax=axes[1,1],color='mediumpurple'); axes[1,1].set_title('Transaction count by payment method'); axes[1,1].set_xlabel('Transactions'); axes[1,1].set_ylabel('Method')
plt.tight_layout(); plt.show()

## 8. Monthly trend

This chart shows whether total spend and transaction volume change over time.

In [ ]:
trend=monthly.reset_index()
fig,ax1=plt.subplots(figsize=(16,5))
sns.lineplot(data=trend,x='month',y='total_spend',marker='o',ax=ax1,color='steelblue'); ax1.set_title('Monthly total spend and transaction count'); ax1.set_xlabel('Month'); ax1.set_ylabel('Total spend ($)'); ax1.tick_params(axis='x',rotation=45)
ax2=ax1.twinx(); sns.lineplot(data=trend,x='month',y='transactions',marker='s',ax=ax2,color='coral'); ax2.set_ylabel('Transaction count')
plt.tight_layout(); plt.show()
display(trend.sort_values('total_spend',ascending=False).head(5).style.format({'total_spend':'${:,.2f}'}))

## Beginner conclusion checklist

Write your conclusions by answering: Which category and customer spend the most? What is the high-value threshold? Which weekday and hour are busiest? Which month has the highest spend? Use the numbers printed above to support each answer.